# 04 — Visualization: SDSS DR17 Stellar Classification
**DSA 1080A · Group 10 · Project 10 — Space Observation and Satellite Data Trends**

Week 4: six Matplotlib charts (bar, time-series line, histogram, category-share, scatter, box plot), each exported to `visuals/` as a PNG with a short interpretation underneath.

Run order: `01_foundations` → `02_exploration` → `03_cleaning_analysis` → **`04_visualization`**.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("../visuals", exist_ok=True)

CLASS_ORDER  = ["GALAXY", "STAR", "QSO"]
CLASS_COLORS = {"GALAXY": "#1f77b4", "STAR": "#ff7f0e", "QSO": "#2ca02c"}
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 200, "font.size": 11,
                     "axes.titlesize": 13, "axes.labelsize": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

def save_fig(fig, name):
    path = f"../visuals/{name}.png"
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print("saved", path)

## Load data
Loads the cleaned dataset saved by `03_cleaning_analysis.ipynb` (99,999 rows x 28 columns). Run notebook 03 first so the file exists.

In [ ]:
PROCESSED = "../data/processed/star_classification_clean.csv"   # saved by 03_cleaning_analysis.ipynb

df = pd.read_csv(PROCESSED)
df["obs_date"] = pd.to_datetime(df["obs_date"])

print(df.shape)
df[["class", "redshift", "r", "alpha", "delta", "obs_year"]].head()

## Figure 1 — Category share of the sample (horizontal bar)

In [ ]:
share = df["class"].value_counts(normalize=True).reindex(CLASS_ORDER) * 100
counts = df["class"].value_counts().reindex(CLASS_ORDER)

fig, ax = plt.subplots(figsize=(8, 3.6))
bars = ax.barh(share.index, share.values, color=[CLASS_COLORS[c] for c in share.index])
ax.invert_yaxis()
ax.set_xlabel("Share of records (%)")
ax.set_ylabel("Object class")
ax.set_title("Galaxies make up about 59% of the SDSS DR17 sample")
ax.set_xlim(0, 70)
for b, pct, n in zip(bars, share.values, counts.values):
    ax.text(b.get_width() + 1, b.get_y() + b.get_height()/2, f"{pct:.1f}%  (n = {n:,})", va="center")
save_fig(fig, "fig1_class_share")
plt.show()

**Figure 1. Share of records by object class (n = 99,999).**
Galaxies make up 59.4% of the sample, stars 21.6% and quasars (QSO) 19.0%, so galaxies outnumber each other class roughly three to one. This matters because any overall average (for example mean brightness) is dominated by galaxies unless it is calculated per class. A horizontal bar chart was used instead of a pie chart because the labels and counts are easier to read and compare. Caveat: this is the mix of objects SDSS chose to take spectra of, not the mix of objects in the real sky, so the shares should not be read as cosmic abundances.

## Figure 2 — Mean redshift by class (comparison bar chart)

In [ ]:
mean_z = df.groupby("class")["redshift"].mean().reindex(CLASS_ORDER)

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(mean_z.index, mean_z.values, color=[CLASS_COLORS[c] for c in mean_z.index])
ax.set_xlabel("Object class")
ax.set_ylabel("Mean redshift (z, unitless)")
ax.set_title("Quasars have by far the highest mean redshift")
ax.set_ylim(0, 2.1)
for b, v in zip(bars, mean_z.values):
    ax.text(b.get_x() + b.get_width()/2, max(v, 0) + 0.04, f"{v:.2f}", ha="center")
save_fig(fig, "fig2_mean_redshift_by_class")
plt.show()

**Figure 2. Mean redshift by object class.**
Quasars average a redshift of 1.72, galaxies 0.42, and stars essentially 0 (about -0.0001). Higher redshift means the light has travelled further and is coming from a more distant object, so this ordering shows stars are inside our own galaxy, ordinary galaxies are at moderate distances, and quasars are among the most distant objects observed. This is the clearest single separation between the classes in the data. Caveat: a mean hides spread, and 118 quasars with redshift above 6 were flagged in Week 3 as probably fitting-software limits (many share exactly 7.011245), so the quasar mean may be slightly overstated. Redshift is also only a distance proxy, so Figure 3 shows the full distributions.

## Figure 3 — Redshift distribution (histogram)

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
bins = [i * 0.1 for i in range(-1, 71)]
for c in CLASS_ORDER:
    ax.hist(df.loc[df["class"] == c, "redshift"], bins=bins, alpha=0.65,
            label=c, color=CLASS_COLORS[c])
ax.set_yscale("log")
ax.set_xlabel("Redshift (z, unitless)")
ax.set_ylabel("Number of objects (log scale)")
ax.set_title("Redshift distribution: stars at 0, galaxies below ~1, quasars spread out to 7")
ax.legend(title="Class")
save_fig(fig, "fig3_redshift_histogram")
plt.show()

**Figure 3. Distribution of redshift by class (bin width 0.1, log-scaled y-axis).**
Essentially all stars sit in the two bins around zero, most galaxies lie between 0 and 1 (median 0.46), and quasars form a wide spread with a median of 1.62 and a tail out to 7.0 (the values near 7.01 were flagged in Week 3 as probable fitting limits). About 80% of quasars have z > 1, compared with only 1.5% of galaxies, so redshift alone almost separates the two extragalactic classes. A log y-axis was needed because the galaxy counts are far larger than the quasar tail and would hide it on a linear scale. Caveat: the bins overlap between 0.5 and 2, where a galaxy and a quasar can have similar redshift, so redshift alone is not a complete classifier.

## Figure 4 — Records observed per year, by class (time-series line chart)

In [ ]:
by_year = (df.groupby(["obs_year", "class"]).size().unstack("class")
             .reindex(columns=CLASS_ORDER).fillna(0))

fig, ax = plt.subplots(figsize=(10, 5))
for c in CLASS_ORDER:
    ax.plot(by_year.index, by_year[c], marker="o", markersize=4, label=c, color=CLASS_COLORS[c])
ax.set_xlabel("Observation year (from MJD)")
ax.set_ylabel("Number of records")
ax.set_title("Records observed per year: galaxy-heavy 2010-2013, quasar-heavy after 2014")
ax.set_xticks(range(2000, 2021, 2))
ax.grid(axis="y", alpha=0.3)
ax.legend(title="Class")
save_fig(fig, "fig4_records_per_year")
plt.show()

**Figure 4. Number of records observed per year, split by class.**
Observations span March 2000 to March 2020 and peak in 2011-2012 at about 11,000 records a year, almost all of it galaxies. After 2014 the mix changes: quasars rise from about 16% of the year's records in 2010 to about 44% in 2018, and galaxies fall. This suggests the survey's targeting changed over time, which matters for anyone using the data to describe long-term trends. Caveat: the counts reflect what was observed in each survey phase, not how many objects exist, and the dips in 2009 and 2019-2020 are likely gaps between phases, not real changes in the sky.

## Figure 5 — Sky positions (scatter plot)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for c in CLASS_ORDER:
    sub = df[df["class"] == c].sample(n=min(6000, (df["class"] == c).sum()), random_state=42)
    ax.scatter(sub["alpha"], sub["delta"], s=3, alpha=0.5, label=f"{c} (6,000 sampled)", color=CLASS_COLORS[c])
ax.set_xlabel("Right ascension, alpha (degrees)")
ax.set_ylabel("Declination, delta (degrees)")
ax.set_title("Sky positions: SDSS observes specific regions, not the whole sky")
ax.set_xlim(0, 360)
ax.legend(markerscale=4, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, frameon=False)
save_fig(fig, "fig5_sky_positions")
plt.show()

**Figure 5. Sky positions (right ascension vs declination) for a random sample of 6,000 objects per class.**
Over half of all objects (about 56,500) lie between alpha = 120 and 240 degrees, in one large arch-shaped region, and 88% are in the northern sky (delta >= 0), with two further blocks at the left and right edges (alpha below 60 and above 310) and large empty gaps elsewhere. Galaxies and quasars follow the same footprint, while stars also appear in scattered patches outside it, so the classes differ only slightly in where they are found. For survey planners this shows where coverage is dense and where large gaps remain. Caveat: the plot is a flat projection of a sphere, so areas near high declination are stretched, and sampling 6,000 per class makes the classes look equally common when they are not (see Figure 1).

## Figure 6 — r-band brightness by class (box plot)

In [ ]:
data = [df.loc[df["class"] == c, "r"] for c in CLASS_ORDER]

fig, ax = plt.subplots(figsize=(7.5, 5))
bp = ax.boxplot(data, patch_artist=True, showfliers=False,
                medianprops={"color": "black"})
for patch, c in zip(bp["boxes"], CLASS_ORDER):
    patch.set_facecolor(CLASS_COLORS[c])
    patch.set_alpha(0.7)
ax.set_xticks([1, 2, 3], CLASS_ORDER)
ax.set_xlabel("Object class")
ax.set_ylabel("r-band magnitude (smaller = brighter)")
ax.set_title("Stars are brightest in the r band; quasars are faintest")
ax.invert_yaxis()
ax.grid(axis="y", alpha=0.3)
save_fig(fig, "fig6_rband_by_class")
plt.show()

**Figure 6. r-band magnitude by class (outliers hidden; y-axis inverted so brighter is higher).**
Median r magnitudes are 18.96 for stars, 20.10 for galaxies and 20.77 for quasars, so stars are the brightest class and quasars the faintest. Magnitudes run backwards (a smaller number is a brighter object), which is why the axis is inverted. Quasars are faint despite being intrinsically luminous because they are extremely distant, which links back to Figure 2. Caveat: apparent brightness depends on both distance and true luminosity, so this chart cannot say which class is intrinsically brightest.

## Export check
Confirm that every figure was written to `visuals/`.

In [ ]:
for f in sorted(os.listdir("../visuals")):
    print(f)